# Detection: YOLOv11s [Dairy Cow Dataset]-Final

## Step 0A: Runpod login and notebook connectivity

In [ ]:
# Step 0A helper-step: setting Runpod dataset root if needed
import os
from pathlib import Path

RUNPOD_NOTEBOOK_ROOT = Path(os.environ.get('RUNPOD_NOTEBOOK_ROOT', '/workspace/*'))
RUNPOD_DATA_ROOT = Path(os.environ.get('RUNPOD_DATA_ROOT', '/workspace/**'))
LOCAL_DATA_ROOT = Path('C:/Users/User/**/**/**')

print('Runpod notebook root:', RUNPOD_NOTEBOOK_ROOT)
print('Runpod dataset root :', RUNPOD_DATA_ROOT)
print('Local dataset root  :', LOCAL_DATA_ROOT)

## Step 0: Packages installation

In [ ]:
import sys

print(f'Python executable: {sys.executable}')

!python -m pip install --upgrade pip setuptools wheel
!python -m pip install -U ultralytics
!python -m pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!python -m pip install -U opencv-python pyyaml pandas numpy matplotlib psutil seaborn pillow jupyterlab ipykernel tqdm

try:
    import torch
    print('Torch version:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
except Exception as exc:
    print('Torch check failed:', exc)

## Step 1: Libraries

In [ ]:
import os
import re
import gc
import glob
import yaml
import json
import time
import math
import shutil
import random
import platform
import psutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image

import torch
from ultralytics import YOLO
from IPython.display import display, Image as IPyImage

import warnings
warnings.filterwarnings("ignore")

## Step 2A: Dataset and output paths


In [ ]:
ORIG_ROOT = Path.cwd()
LOCAL_DAIRY_ROOT = LOCAL_DATA_ROOT
RUNPOD_DAIRY_ROOT = RUNPOD_DATA_ROOT
DEFAULT_RUNS_ROOT = ORIG_ROOT / 'out_det'
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def looks_like_dataset(root: Path) -> bool:
    return (
        root.exists()
        and (root / 'images' / 'train').exists()
        and (root / 'labels' / 'train').exists()
        and (root / 'images' / 'val').exists()
        and (root / 'labels' / 'val').exists()
    )


def find_dataset_root() -> Path | None:
    candidates = [
        LOCAL_DAIRY_ROOT,
        RUNPOD_DAIRY_ROOT,
        ORIG_ROOT,
        ORIG_ROOT.parent,
        Path('/workspace'),
        Path('/workspace/*'),
        Path('/workspace/**'),
        Path('/workspace/**'),
        Path('/root'),
    ]

    for candidate in candidates:
        if looks_like_dataset(candidate):
            return candidate

    search_roots = [Path('/workspace'), ORIG_ROOT, ORIG_ROOT.parent]
    for base in search_roots:
        if not base.exists():
            continue
        try:
            for p in base.rglob('*'):
                if p.is_dir() and looks_like_dataset(p):
                    return p
        except Exception:
            continue
    return None


DAIRY_ROOT = find_dataset_root()
RUNS_ROOT = Path(os.environ.get('YOLO_RUNS_ROOT', str(DEFAULT_RUNS_ROOT)))
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

print('Current notebook directory:', ORIG_ROOT.resolve())
print('Requested Runpod dataset root:', RUNPOD_DAIRY_ROOT)
print('Runs root:', RUNS_ROOT.resolve())

if DAIRY_ROOT is None:
    print('\nDataset root was not found automatically.')
    print('Put the dataset in one of these locations:')
    print('  /workspace/**')
    print('  /workspace/**/**')
    print('  a folder that contains images/train, labels/train, images/val, labels/val')

    workspace = Path('/workspace')
    if workspace.exists():
        print('\nTop-level folders under /workspace:')
        for p in sorted(workspace.iterdir()):
            if p.is_dir():
                print(' ', p)

    raise FileNotFoundError(
        'Dataset root not found. Upload the Dairy Cow dataset to /workspace/**, then rerun Step 2A.'
    )

orig_yaml = DAIRY_ROOT / 'data.yaml'
full_yaml = DAIRY_ROOT / 'data_full_detection.yaml'

train_img_dir = DAIRY_ROOT / 'images' / 'train'
val_img_dir = DAIRY_ROOT / 'images' / 'val'
buffer_img_dir = DAIRY_ROOT / 'images' / 'buffer'

train_lbl_dir = DAIRY_ROOT / 'labels' / 'train'
val_lbl_dir = DAIRY_ROOT / 'labels' / 'val'
buffer_lbl_dir = DAIRY_ROOT / 'labels' / 'buffer'

print('\nSelected dataset root:', DAIRY_ROOT.resolve())
print('Original data.yaml exists:', orig_yaml.exists())
print('train/images exists:', train_img_dir.exists())
print('train/labels exists:', train_lbl_dir.exists())
print('val/images exists:', val_img_dir.exists())
print('val/labels exists:', val_lbl_dir.exists())
print('buffer/images exists:', buffer_img_dir.exists())
print('buffer/labels exists:', buffer_lbl_dir.exists())

assert train_img_dir.exists(), f'Missing folder: {train_img_dir}'
assert train_lbl_dir.exists(), f'Missing folder: {train_lbl_dir}'
assert val_img_dir.exists(), f'Missing folder: {val_img_dir}'
assert val_lbl_dir.exists(), f'Missing folder: {val_lbl_dir}'

all_images = sorted([p for p in train_img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])
all_labels = sorted(list(train_lbl_dir.glob('*.txt')))

print(f'\nTotal training images found: {len(all_images)}')
print(f'Total training labels found: {len(all_labels)}')

data_cfg = {
    'path': str(DAIRY_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 5,
    'names': ['Drinking', 'Feeding', 'Resting', 'Standing', 'Walking'],
}

with open(full_yaml, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print('\nCreated YAML file:')
print(full_yaml)

with open(full_yaml, 'r', encoding='utf-8') as f:
    loaded_cfg = yaml.safe_load(f)

print('\nContents of data_full_detection.yaml:')
print(loaded_cfg)

names = loaded_cfg['names']
if isinstance(names, dict):
    class_map = {int(k): v for k, v in names.items()}
else:
    class_map = {i: v for i, v in enumerate(names)}

print('\nClass map:')
for k, v in class_map.items():
    print(f'  {k}: {v}')

## Step 2B: Inspecting a few image-label pairs

In [ ]:
def find_label_for_image(img_path: Path, label_dir: Path):
    return label_dir / f"{img_path.stem}.txt"

sample_images = all_images[:5]

for img_path in sample_images:
    lbl_path = find_label_for_image(img_path, train_lbl_dir)
    print(f"\nImage: {img_path.name}")
    print(f"Label: {lbl_path.name} | Exists: {lbl_path.exists()}")
    if lbl_path.exists():
        with open(lbl_path, "r", encoding="utf-8") as f:
            lines = f.readlines()[:5]
        print("First few label lines:")
        for line in lines:
            print("  ", line.strip())

## STEP 3A: Machine/Laptop or Runpod Pod Specifications


In [ ]:
print("System Information")
print("OS:", platform.system(), platform.release())
print("Python version:", platform.python_version())
print("Processor:", platform.processor())
print("Machine:", platform.machine())

print("\nCPU/RAM")
print("Physical cores:", psutil.cpu_count(logical=False))
print("Logical cores:", psutil.cpu_count(logical=True))
print("Total RAM (GB):", round(psutil.virtual_memory().total / (1024**3), 2))
print("Available RAM:", round(psutil.virtual_memory().available / (1024**3), 2))

print("\nStorage")
disk = psutil.disk_usage("/")
print("Total disk (GB):", round(disk.total / (1024**3), 2))
print("Used disk (GB):", round(disk.used / (1024**3), 2))
print("Free disk (GB):", round(disk.free / (1024**3), 2))

print("\nPYTORCH / CUDA")
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU device:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("GPU memory (GB):", round(props.total_memory / (1024**3), 2))
else:
    print("GPU device: CPU only")

## Step 3B: GPU check and Runpod readiness


In [ ]:
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
else:
    gpu_mem_gb = 0

if gpu_mem_gb >= 14:
    recommended_batch = 16
    recommended_imgsz = 640
    recommended_model = 'yolo11s.pt'
    recommended_workers = 8
    recommended_epochs = 40
    recommended_patience = 10
elif gpu_mem_gb >= 7.5:
    recommended_batch = 8
    recommended_imgsz = 640
    recommended_model = 'yolo11s.pt'
    recommended_workers = 4
    recommended_epochs = 40
    recommended_patience = 10
else:
    recommended_batch = 2
    recommended_imgsz = 640
    recommended_model = 'yolo11n.pt'
    recommended_workers = 2
    recommended_epochs = 40
    recommended_patience = 10

print('Recommended model:', recommended_model)
print('Recommended batch:', recommended_batch)
print('Recommended imgsz:', recommended_imgsz)
print('Recommended workers:', recommended_workers)
print('Recommended epochs:', recommended_epochs)
print('Recommended patience:', recommended_patience)

## STEP 4A: Full dataset structure inspection

In [ ]:
for split in ["train", "val", "buffer"]:
    img_dir = DAIRY_ROOT / "images" / split
    lbl_dir = DAIRY_ROOT / "labels" / split
    print(f"\n{split.upper()}")
    print("Images:", len([p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS]))
    print("Labels:", len(list(lbl_dir.glob("*.txt"))))

## Step 4B: Full dataset distribution structure

In [ ]:
def get_light_condition(filename):
    name = filename.lower()
    if "night" in name:
        return "night"
    elif "day" in name:
        return "day"
    else:
        return "unknown"

def extract_date_from_filename(filename):
    match = re.search(r"(20\d{6})", filename)
    if match:
        return match.group(1)
    return "unknown"

def extract_hour_from_filename(filename):
    match = re.search(r"(20\d{6})(\d{2})(\d{2})(\d{2})", filename)
    if match:
        return match.group(2)
    return "unknown"

def read_label_classes(label_path):
    if not label_path.exists():
        return []
    cls_ids = []
    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 1:
                try:
                    cls_ids.append(int(float(parts[0])))
                except:
                    pass
    return cls_ids

rows = []
for split in ["train", "val", "buffer"]:
    img_dir = DAIRY_ROOT / "images" / split
    lbl_dir = DAIRY_ROOT / "labels" / split

    for img_path in sorted(img_dir.iterdir()):
        if img_path.suffix.lower() not in IMAGE_EXTS:
            continue

        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        cls_ids = read_label_classes(lbl_path)

        row = {
            "split": split,
            "filename": img_path.name,
            "light_condition": get_light_condition(img_path.name),
            "date": extract_date_from_filename(img_path.name),
            "hour": extract_hour_from_filename(img_path.name),
            "num_instances": len(cls_ids),
            "classes_in_image": sorted(set(cls_ids)),
        }

        for class_id, class_name in class_map.items():
            row[f"has_{class_name}"] = int(class_id in cls_ids)
            row[f"count_{class_name}"] = cls_ids.count(class_id)

        rows.append(row)

df_audit = pd.DataFrame(rows)
print("Total full-dataset images inspected:", len(df_audit))
display(df_audit.head())

split_table = df_audit["split"].value_counts().rename_axis("split").reset_index(name="images")
light_table = df_audit["light_condition"].value_counts().rename_axis("light_condition").reset_index(name="images")
date_table = df_audit["date"].value_counts().sort_index().rename_axis("date").reset_index(name="images")
hour_table = df_audit["hour"].value_counts().sort_index().rename_axis("hour").reset_index(name="images")

print("\n1) Split distribution")
display(split_table)

print("\n2) Day and Nighttime distribution")
display(light_table)

print("\n3) Full dataset date distribution")
display(date_table)

print("\n4) Full dataset hourly distribution")
display(hour_table)

split_light = pd.crosstab(df_audit["split"], df_audit["light_condition"])
split_date = pd.crosstab(df_audit["split"], df_audit["date"])
split_hour = pd.crosstab(df_audit["split"], df_audit["hour"])

print("\n5) Split vs. Light condition")
display(split_light)

print("\n6) Split vs. Date")
display(split_date)

print("\n7) Split vs. Hour")
display(split_hour)

image_presence_rows = []
for class_id, class_name in class_map.items():
    col = f"has_{class_name}"
    total_images = int(df_audit[col].sum())
    image_presence_rows.append({
        "class_id": class_id,
        "class_name": class_name,
        "images_containing_class": total_images
    })
df_image_presence = pd.DataFrame(image_presence_rows).sort_values(
    "images_containing_class", ascending=False
)

print("\n8) Class distribution by image presence")
display(df_image_presence)

instance_rows = []
for class_id, class_name in class_map.items():
    col = f"count_{class_name}"
    total_instances = int(df_audit[col].sum())
    instance_rows.append({
        "class_id": class_id,
        "class_name": class_name,
        "total_instances": total_instances
    })
df_instances = pd.DataFrame(instance_rows).sort_values(
    "total_instances", ascending=False
)

print("\n9) Class distribution by total number of object instances")
display(df_instances)

split_class_rows = []
for split in ["train", "val", "buffer"]:
    df_split = df_audit[df_audit["split"] == split]
    row = {"split": split}
    for class_id, class_name in class_map.items():
        row[class_name] = int(df_split[f"count_{class_name}"].sum())
    split_class_rows.append(row)

df_split_class_instances = pd.DataFrame(split_class_rows)
print("\n10) Split vs. class instance counts")
display(df_split_class_instances)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

plt.figure()
sns.barplot(data=split_table, x="split", y="images")
plt.title("Full Dataset Split Distribution")
plt.xlabel("Split")
plt.ylabel("Number of Images")
plt.show()

plt.figure()
sns.barplot(data=light_table, x="light_condition", y="images")
plt.title("Day vs. Night Distribution")
plt.xlabel("Light condition")
plt.ylabel("Number of Images")
plt.show()

plt.figure(figsize=(12, 6))
sns.barplot(data=hour_table, x="hour", y="images")
plt.title("Hourly Images Distribution")
plt.xlabel("Hours")
plt.ylabel("Number of Images")
plt.show()

plt.figure()
sns.barplot(data=df_image_presence, x="class_name", y="images_containing_class")
plt.title("Images Containing Each Behavior")
plt.xlabel("Behavior Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=20)
plt.show()

plt.figure()
sns.barplot(data=df_instances, x="class_name", y="total_instances")
plt.title("Total Object Instances per Behavior")
plt.xlabel("Behavior Classes")
plt.ylabel("Number of Instances")
plt.xticks(rotation=20)
plt.show()

## Step 4C: RAM memory clean-up

In [ ]:
# Step 4C: Robust memory cleanup before training
import os
import gc
import time
import ctypes
import platform
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

print("=" * 70)
print("ROBUST MEMORY CLEANUP BEFORE TRAINING")
print("=" * 70)

# 1) Close all matplotlib figures
try:
    plt.close("all")
    print("[OK] Closed matplotlib figures.")
except Exception as e:
    print(f"[WARN] Could not close matplotlib figures: {e}")

# 2) Remove large variables created in earlier audit/inspection steps
large_vars = [
    "df_audit",
    "split_light",
    "split_date",
    "split_hour",
    "hour_table",
    "class_count_df",
    "split_class_counts",
    "sample_images",
    "train_imgs",
    "val_imgs",
    "buffer_imgs",
    "train_lbls",
    "val_lbls",
    "buffer_lbls",
    "all_images",
    "all_labels",
    "rows",
    "img_paths",
    "lbl_paths"
]

removed = []
for var_name in large_vars:
    if var_name in globals():
        try:
            del globals()[var_name]
            removed.append(var_name)
        except Exception as e:
            print(f"[WARN] Could not delete {var_name}: {e}")

if removed:
    print(f"[OK] Removed variables: {', '.join(removed)}")
else:
    print("[INFO] No large tracked variables found to remove.")

# 3) Python garbage collection
gc.collect()
print("[OK] Python garbage collection completed.")

# 4) CUDA cache cleanup
if "torch" in globals():
    try:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            print("[OK] CUDA cache cleared.")
        else:
            print("[INFO] CUDA not available.")
    except Exception as e:
        print(f"[WARN] CUDA cleanup issue: {e}")
else:
    print("[INFO] torch not loaded yet; skipped CUDA cleanup.")

# 5) Optional Windows working-set trim
try:
    if platform.system().lower() == "windows":
        ctypes.windll.psapi.EmptyWorkingSet(ctypes.windll.kernel32.GetCurrentProcess())
        print("[OK] Windows working-set trim requested.")
except Exception as e:
    print(f"[WARN] Working-set trim not applied: {e}")

# 6) Final status
print("-" * 70)
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f"RAM used      : {vm.used / (1024**3):.2f} GB")
    print(f"RAM available : {vm.available / (1024**3):.2f} GB")
    print(f"RAM percent   : {vm.percent:.1f}%")
except Exception as e:
    print(f"[WARN] Could not read RAM stats: {e}")

if "torch" in globals():
    try:
        if torch.cuda.is_available():
            print(f"GPU allocated : {torch.cuda.memory_allocated() / (1024**3):.2f} GB")
            print(f"GPU reserved  : {torch.cuda.memory_reserved() / (1024**3):.2f} GB")
    except Exception as e:
        print(f"[WARN] Could not read GPU stats: {e}")

print("=" * 70)
print("Cleanup completed. Safe to proceed to training.")
print("=" * 70)

time.sleep(2)

## Step 4D

In [ ]:
# Step 4D: Inspect current notebook PID and other Python processes safely
import os
import sys
import psutil
from pathlib import Path

print("=" * 80)
print("DETAILED MEMORY INSPECTION")
print("=" * 80)

# Current notebook process
current_proc = psutil.Process(os.getpid())
current_mem = current_proc.memory_info()

print(f"Current notebook PID      : {current_proc.pid}")
print(f"Python executable         : {sys.executable}")
print(f"Notebook RSS memory       : {current_mem.rss / (1024**3):.2f} GB")
print(f"Notebook VMS memory       : {current_mem.vms / (1024**3):.2f} GB")

vm = psutil.virtual_memory()
print(f"System RAM used           : {vm.used / (1024**3):.2f} GB")
print(f"System RAM available      : {vm.available / (1024**3):.2f} GB")
print(f"System RAM percent        : {vm.percent:.1f}%")

print("\n" + "=" * 80)
print("PYTHON PROCESSES")
print("=" * 80)

python_rows = []

for p in psutil.process_iter(["pid", "name", "exe", "cmdline", "memory_info"]):
    try:
        name = (p.info["name"] or "").lower()
        exe = str(p.info["exe"] or "")
        cmdline = " ".join(p.info["cmdline"] or [])
        
        if "python" in name or "python" in exe.lower():
            rss_gb = p.info["memory_info"].rss / (1024**3)
            python_rows.append({
                "pid": p.info["pid"],
                "name": p.info["name"],
                "rss_gb": rss_gb,
                "exe": exe,
                "cmdline": cmdline
            })
    except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
        continue

python_rows = sorted(python_rows, key=lambda x: x["rss_gb"], reverse=True)

for row in python_rows:
    flag = " <-- CURRENT NOTEBOOK" if row["pid"] == current_proc.pid else ""
    print(f"PID={row['pid']:<8} RAM={row['rss_gb']:.2f} GB  NAME={row['name']}{flag}")
    print(f"    EXE    : {row['exe']}")
    print(f"    CMDLINE: {row['cmdline'][:200]}")
    print("-" * 80)

print("\n" + "=" * 80)
print("SAFE CMD TO KILL ALL OTHER PYTHON PROCESSES")
print("=" * 80)
print(f'for /f "tokens=2" %a in (\'tasklist ^| findstr python.exe\') do @if not "%a"=="{current_proc.pid}" taskkill /PID %a /F')
print("=" * 80)
print("IMPORTANT: This command keeps the current notebook PID alive and kills other python.exe processes only.")
print("=" * 80)

## Step 5: Training of full Dairy Cow dataset

In [ ]:
# Step 5: Training
DATA_YAML = str(full_yaml)
PROJECT_DIR = str(RUNS_ROOT)
RUN_NAME = os.environ.get('YOLO_RUN_NAME', 'y11_det_v1')
MODEL_NAME = recommended_model
EPOCHS = recommended_epochs
IMGSZ = recommended_imgsz
BATCH = recommended_batch
WORKERS = recommended_workers
PATIENCE = recommended_patience
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

torch.cuda.empty_cache()
gc.collect()

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print('Model:', MODEL_NAME)
print('Data:', DATA_YAML)
print('Project:', PROJECT_DIR)
print('Run name:', RUN_NAME)
print('Device:', DEVICE)
print('Epochs:', EPOCHS)
print('Img size:', IMGSZ)
print('Batch:', BATCH)
print('Workers:', WORKERS)
print('Patience:', PATIENCE)

model = YOLO(MODEL_NAME)

train_results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    patience=PATIENCE,
    device=DEVICE,
    pretrained=True,
    optimizer='auto',
    cache=False,
    amp=True,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=False,
    plots=True,
    verbose=True,
    save=True,
    save_period=5,
    mosaic=1.0,
    mixup=0.0,
    close_mosaic=10,
    fliplr=0.5,
    degrees=5.0,
    translate=0.05,
    scale=0.20,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
)

print('Training completed.')
print('Run directory:', train_results.save_dir)

## Step 6: Evaluation on train and validation

In [ ]:
# Step 6: Evaluation
RUN_DIR = Path(PROJECT_DIR) / RUN_NAME
BEST_PT = RUN_DIR / "weights" / "best.pt"  # best.pt file
LAST_PT = RUN_DIR / "weights" / "last.pt"  # last.pt file

print("Run dir:", RUN_DIR)
print("Best pt:", BEST_PT.exists())
print("Last pt:", LAST_PT.exists())

assert BEST_PT.exists(), f"Missing best.pt at {BEST_PT}"

best_model = YOLO(str(BEST_PT))

split_metrics = {}
EVAL_ROOT = RUN_DIR / "eval"
EVAL_ROOT.mkdir(parents=True, exist_ok=True)

split_eval_dirs = {
    "train": "train_eval",
    "val": "val_eval"
}

for split_name in ["train", "val"]:
    print(f"\nEvaluating split: {split_name}")

    metrics_obj = best_model.val(
        data=DATA_YAML,
        split=split_name,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        plots=True,
        project=str(EVAL_ROOT),
        name=split_eval_dirs[split_name]
    )

    precision = float(metrics_obj.results_dict.get("metrics/precision(B)", np.nan))
    recall = float(metrics_obj.results_dict.get("metrics/recall(B)", np.nan))
    map50 = float(metrics_obj.box.map50)
    map75 = float(metrics_obj.box.map75)
    map5095 = float(metrics_obj.box.map)
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else np.nan

    split_metrics[split_name] = {
        "precision": precision,
        "recall": recall,
        "f1": float(f1) if not np.isnan(f1) else np.nan,
        "mAP50": map50,
        "mAP75": map75,
        "mAP50_95": map5095,
        "per_class_mAP50_95": [float(x) for x in metrics_obj.box.maps]
    }

    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-score  : {f1:.4f}")
    print(f"mAP50     : {map50:.4f}")
    print(f"mAP75     : {map75:.4f}")
    print(f"mAP50-95  : {map5095:.4f}")

df_split_metrics = pd.DataFrame(split_metrics).T.reset_index().rename(columns={"index": "split"})
print("\nYOLOv11s split-wise evaluation summary")
display(df_split_metrics)

metrics_csv_path = RUN_DIR / "split_metrics_yolo11s_full.csv"
metrics_json_path = RUN_DIR / "split_metrics_yolo11s_full.json"

df_split_metrics.to_csv(metrics_csv_path, index=False)
with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(split_metrics, f, indent=4)

print("Saved split metrics CSV to:", metrics_csv_path)
print("Saved split metrics JSON to:", metrics_json_path)

## Step 7: Visualization

In [ ]:
plot_files = [
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxP_curve.png",
    "BoxPR_curve.png",
    "BoxR_curve.png",
    "BoxF1_curve.png",
    "labels.jpg",
    "labels_correlogram.jpg"
]

for plot_name in plot_files:
    plot_path = RUN_DIR / plot_name
    if plot_path.exists():
        print(f"\nDisplaying: {plot_name}")
        display(IPyImage(filename=str(plot_path), width=900))
    else:
        print("Missing:", plot_path)

## Step 8: Visualization of Training curves via results.csv

In [ ]:
# Step 8
results_csv = RUN_DIR / "results.csv"
assert results_csv.exists(), f"results.csv not found: {results_csv}"

df = pd.read_csv(results_csv)
print("Loaded:", results_csv)
display(df.head())

candidate_cols = [
    "train/box_loss", "train/cls_loss", "train/dfl_loss",
    "val/box_loss", "val/cls_loss", "val/dfl_loss",
    "metrics/precision(B)", "metrics/recall(B)",
    "metrics/mAP50(B)", "metrics/mAP50-95(B)"
]

available_cols = [c for c in candidate_cols if c in df.columns]

print("Available columns:")
for c in available_cols:
    print(" ", c)

if available_cols:
    ncols = 3
    nrows = math.ceil(len(available_cols) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax, col in zip(axes, available_cols):
        ax.plot(df.index + 1, df[col], marker="o", linewidth=1.5)
        ax.set_title(col, fontsize=11)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(col)
        ax.grid(True)

    for ax in axes[len(available_cols):]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.show()

metric_col = "metrics/mAP50-95(B)"
if metric_col in df.columns:
    best_epoch = int(df[metric_col].idxmax()) + 1
    best_score = float(df[metric_col].max())
    print(f"\nBest epoch based on {metric_col}: {best_epoch}")
    print(f"Best score: {best_score:.4f}")

## Step 9: Saving experiment summary

In [ ]:
# Step 9
summary = {
    'run_name': RUN_NAME,
    'model': MODEL_NAME,
    'epochs': EPOCHS,
    'imgsz': IMGSZ,
    'batch': BATCH,
    'workers': WORKERS,
    'patience': PATIENCE,
    'device': str(DEVICE),
    'dataset_root': str(DAIRY_ROOT.resolve()),
    'data_yaml': str(full_yaml.resolve()),
    'best_weights': str(BEST_PT.resolve()),
    'last_weights': str(LAST_PT.resolve()) if LAST_PT.exists() else None,
    'results_csv': str(results_csv.resolve()),
    'train_metrics': split_metrics['train'],
    'val_metrics': split_metrics['val'],
}

summary_path = RUN_DIR / 'sum_det.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=4)

print('Saved summary to:', summary_path)
print(json.dumps(summary, indent=4))

## Step 10: Comparison template row

In [ ]:
# Step 10
comparison_row = pd.DataFrame([{
    'model': MODEL_NAME,
    'run_name': RUN_NAME,
    'train_precision': split_metrics['train']['precision'],
    'train_recall': split_metrics['train']['recall'],
    'train_f1': split_metrics['train']['f1'],
    'train_mAP50': split_metrics['train']['mAP50'],
    'train_mAP75': split_metrics['train']['mAP75'],
    'train_mAP50_95': split_metrics['train']['mAP50_95'],
    'val_precision': split_metrics['val']['precision'],
    'val_recall': split_metrics['val']['recall'],
    'val_f1': split_metrics['val']['f1'],
    'val_mAP50': split_metrics['val']['mAP50'],
    'val_mAP75': split_metrics['val']['mAP75'],
    'val_mAP50_95': split_metrics['val']['mAP50_95'],
}])

comparison_path = RUN_DIR / 'cmp_det.csv'
comparison_row.to_csv(comparison_path, index=False)

print('Saved comparison row to:', comparison_path)
display(comparison_row)
